# 🔍 Detection-Only Production Notebook
## Pothole/Crack Detection with GPS Sync

---

**Purpose:** Fast detection without segmentation masks.  
**Output:** `detections.csv` + annotated images for web app upload.

## 1️⃣ Setup & GPU Check

In [ ]:
# @title 1️⃣ Setup & Configuration
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Enable GPU: Runtime → Change runtime type → GPU")

# @markdown ### ⚙️ Settings
FRAME_SKIP = 5 # @param {type:"integer"}
CONFIDENCE_THRESHOLD = 0.25 # @param {type:"number"}
IMG_SIZE = 640

print(f"✅ Settings: FRAME_SKIP={FRAME_SKIP}, CONF={CONFIDENCE_THRESHOLD}")

!pip install -U ultralytics opencv-python pandas numpy tqdm

In [ ]:
# @title 2️⃣ Define File Paths
# @markdown Upload files to Colab first, then enter paths.

# @markdown ### 📹 Input Video
VIDEO_PATH = "/content/video.mp4" # @param {type:"string"}

# @markdown ### 🧠 Detection Model
DET_MODEL_PATH = "/content/best_det.pt" # @param {type:"string"}

# @markdown ### 📍 GPS Log
GPS_LOG_PATH = "/content/gps_log.csv" # @param {type:"string"}

import os
for p, n in [(VIDEO_PATH, 'Video'), (DET_MODEL_PATH, 'Model'), (GPS_LOG_PATH, 'GPS')]:
    if os.path.exists(p):
        print(f"✅ {n} found: {p}")
    else:
        print(f"⚠️ {n} NOT found: {p}")

## 3️⃣ Load GPS Data

In [ ]:
import pandas as pd

gps_data = None
video_start_time = None

if os.path.exists(GPS_LOG_PATH):
    try:
        df = pd.read_csv(GPS_LOG_PATH)
        df.columns = df.columns.str.strip().str.lower()
        
        time_col = 'time' if 'time' in df.columns else 'timestamp'
        df[time_col] = pd.to_datetime(df[time_col], format='mixed', utc=True)
        df = df.sort_values(time_col)
        df = df.rename(columns={time_col: 'time'})
        
        gps_data = df
        video_start_time = gps_data['time'].iloc[0]
        print(f"✅ Loaded {len(gps_data)} GPS points")
    except Exception as e:
        print(f"❌ GPS Error: {e}")
else:
    print("❌ GPS file not found!")

## 4️⃣ Load Detection Model

In [ ]:
from ultralytics import YOLO

det_model = None
DET_CLASSES = {}

if os.path.exists(DET_MODEL_PATH):
    det_model = YOLO(DET_MODEL_PATH)
    det_model.to('cuda' if torch.cuda.is_available() else 'cpu')
    DET_CLASSES = det_model.names
    print(f"✅ Model Loaded. Classes: {DET_CLASSES}")
else:
    print(f"❌ Model NOT found at {DET_MODEL_PATH}")

## 5️⃣ Process Video (Detection Only)

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print("❌ Could not open video")
else:
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"✅ Video: {total_frames} frames @ {fps:.1f} FPS")

os.makedirs('results/frames', exist_ok=True)
detection_records = []
frame_idx = 0
detection_count = 0

pbar = tqdm(total=total_frames // FRAME_SKIP, desc="Detecting")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    if frame_idx % FRAME_SKIP == 0:
        # 1. Detection
        results = det_model.predict(frame, conf=CONFIDENCE_THRESHOLD, imgsz=IMG_SIZE, verbose=False)[0]
        
        # 2. GPS Sync
        lat, lon = None, None
        if gps_data is not None and video_start_time is not None:
            video_time = video_start_time + pd.to_timedelta(frame_idx / fps, unit='s')
            closest_idx = (gps_data['time'] - video_time).abs().idxmin()
            gps_row = gps_data.loc[closest_idx]
            lat = float(gps_row['latitude'])
            lon = float(gps_row['longitude'])
        
        # 3. Process Detections
        if results.boxes is not None and len(results.boxes) > 0:
            boxes = results.boxes.xyxy.cpu().numpy()
            scores = results.boxes.conf.cpu().numpy()
            classes = results.boxes.cls.cpu().numpy().astype(int)
            
            annotated_frame = frame.copy()
            
            for i, (box, score, cls) in enumerate(zip(boxes, scores, classes)):
                x1, y1, x2, y2 = box.astype(int)
                class_name = DET_CLASSES[cls]
                
                # Draw Red Box
                color = (0, 0, 255)
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                label = f"{class_name} {score:.2f}"
                cv2.putText(annotated_frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                
                # Save Image
                img_filename = f"detection_{detection_count}.jpg"
                img_path = f"results/frames/{img_filename}"
                cv2.imwrite(img_path, annotated_frame)
                
                # Record
                detection_records.append({
                    'latitude': lat,
                    'longitude': lon,
                    'image_path': img_filename,
                    'confidence_score': float(score),
                    'class': class_name,
                    'box_width': int(x2 - x1),
                    'box_height': int(y2 - y1),
                    'box_center_y': int((y1 + y2) / 2)
                })
                detection_count += 1
        
        pbar.update(1)
    
    frame_idx += 1

cap.release()
pbar.close()
print(f"\n✅ Complete. Found {len(detection_records)} detections.")

## 6️⃣ Save & Download Results

In [ ]:
from google.colab import files
import shutil

if len(detection_records) > 0:
    df = pd.DataFrame(detection_records)
    df.to_csv('results/detections.csv', index=False)
    
    print("📊 Detection Breakdown:")
    print(df['class'].value_counts())
    
    print("\n📦 Zipping results...")
    shutil.make_archive('detection_results', 'zip', 'results')
    files.download('detection_results.zip')
    print("🎉 Downloaded detection_results.zip")
else:
    print("⚠️ No detections to save.")